# Gemma 4 Research Agent

An agentic pipeline built on **Gemma 4 12B** that takes a research question, breaks it into sub-questions, searches the web for each, and synthesizes a structured answer — all orchestrated with plain Python and no frameworks.

**Pipeline:**
```
User Question
      │
      ▼
Planner (Gemma 4)      ← breaks question into 2-3 sub-questions
      │
search_web() × N       ← ddgs fetches results for each sub-question
      │
Synthesizer (Gemma 4)  ← reads all results, writes structured answer
      │
      ▼
Final Answer
```

**Stack:**
| Component | Tool |
|---|---|
| Model | `google/gemma-4-12B-it` via HuggingFace Transformers |
| Quantization | 4-bit NF4 via BitsAndBytes (~7.5 GB VRAM) |
| Web search | `ddgs` (free, no API key) |
| Runtime | Kaggle T4 x2 GPU |

**References:**
- Model card: https://huggingface.co/google/gemma-4-12B-it
- Transformers docs: https://huggingface.co/docs/transformers/main/en/model_doc/gemma4_unified
- ddgs library: https://github.com/deedy5/ddgs
- Google AI Gemma docs: https://ai.google.dev/gemma/docs/core

## Prerequisites

Before running this notebook:

1. **Accept the Gemma 4 license** on HuggingFace: https://huggingface.co/google/gemma-4-12B-it
2. **Create a HuggingFace token** (Read access): https://huggingface.co/settings/tokens
3. **Add the token to Kaggle Secrets** — name it exactly `HF_TOKEN`, toggle Notebook access on
4. **Set accelerator to GPU T4 x2** in notebook Settings
5. **Enable Internet** in notebook Settings

## Step 1 — Install Dependencies

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate ddgs

> **Restart the kernel after installation** before proceeding to the next cell.
> Run → Restart & Clear Outputs, then continue from Step 2.

## Step 2 — Authentication

Reads the HuggingFace token from Kaggle Secrets and sets it as an environment variable.
HuggingFace Transformers automatically reads `HF_TOKEN` from the environment — no `login()` call needed.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

print("Token set:", os.environ["HF_TOKEN"][:8] + "...")

## Step 3 — Load Model

Loads `google/gemma-4-12B-it` in 4-bit NF4 quantization using BitsAndBytes.

- **`device_map="auto"`** distributes the 48 transformer layers across both T4 GPUs automatically
- **4-bit NF4** compresses the model from ~24 GB to ~7.5 GB with minimal quality loss
- **`AutoModelForImageTextToText`** is the correct class for Gemma 4 12B Unified (encoder-free multimodal architecture)
- Model weights are cached after the first download — subsequent loads take under 2 minutes

References:
- `AutoModelForImageTextToText`: https://huggingface.co/docs/transformers/main/en/model_doc/gemma4_unified
- `BitsAndBytesConfig`: https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes

In [ ]:
import re
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-12B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

model.eval()

print("Memory footprint:", round(model.get_memory_footprint() / 1e9, 2), "GB")
print("Device map:", model.hf_device_map)

## Step 4 — Load Agent Modules

Defines all functions from `model.py`, `tools.py`, `prompts.py`, and `agent.py`.
In a local environment these would be imported directly. On Kaggle, we define them inline.

See the project repo for the individual source files:
https://github.com/dwinsi/gemma4-research-agent

In [ ]:
# ── model.py ────────────────────────────────────────────────────────────────

MAX_NEW_TOKENS = 1024
TEMPERATURE    = 0.7
TOP_P          = 0.95

def _strip_thinking_tokens(text: str) -> str:
    """Removes Gemma 4 thinking blocks: <|channel>thought....<channel|>"""
    cleaned = re.sub(
        r"<\|channel>thought.*?<channel\|>",
        "",
        text,
        flags=re.DOTALL
    )
    return cleaned.strip()

def generate_response(
    model,
    processor,
    user_message: str,
    system_message: str = "You are a helpful research assistant."
) -> str:
    """Formats prompt, runs inference, returns clean assistant response."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_message}]},
        {"role": "user",   "content": [{"type": "text", "text": user_message}]}
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_token_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_token_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True
        )

    response_token_ids = output_token_ids[0][input_token_length:]
    raw_response = processor.decode(response_token_ids, skip_special_tokens=True)
    return _strip_thinking_tokens(raw_response).strip()


# ── tools.py ────────────────────────────────────────────────────────────────

from ddgs import DDGS

MAX_SEARCH_RESULTS = 5

def search_web(query: str, max_results: int = MAX_SEARCH_RESULTS) -> list[dict]:
    """Searches the web and returns list of dicts with title, url, snippet."""
    try:
        raw_results = DDGS().text(
            query,
            region="us-en",
            safesearch="moderate",
            max_results=max_results
        )
        return [
            {"title": r.get("title", ""), "url": r.get("href", ""), "snippet": r.get("body", "")}
            for r in raw_results
        ]
    except Exception as error:
        print(f"[search_web] Failed for '{query}': {error}")
        return []

def format_search_results_for_prompt(query: str, results: list[dict]) -> str:
    """Formats search results into a prompt-ready string block."""
    if not results:
        return f'No results found for: "{query}"'
    lines = [f'Search results for: "{query}"\n']
    for index, result in enumerate(results, start=1):
        lines.append(
            f"[{index}] Title: {result['title']}\n"
            f"    URL: {result['url']}\n"
            f"    Snippet: {result['snippet']}\n"
        )
    return "\n".join(lines)


# ── prompts.py ───────────────────────────────────────────────────────────────

PLANNER_SYSTEM_PROMPT = """You are a research planner. Your only job is to break a complex question into 2 or 3 focused sub-questions that can each be answered by a single web search.

Rules you must follow without exception:
- Output ONLY a numbered list. No introduction, no explanation, no closing remarks.
- Each sub-question must be specific and self-contained.
- Each sub-question must be on its own line.
- Do not number beyond 3.
- Do not output anything other than the numbered list.

Example output format:
1. What is the history of quantum computing?
2. What are the latest breakthroughs in quantum computing in 2025?
3. Which companies are leading quantum computing research today?"""

SYNTHESIZER_SYSTEM_PROMPT = """You are a research synthesizer. You receive a research question and a set of web search results. Your job is to write a clear, well-structured answer based strictly on the search results provided.

Rules you must follow:
- Base your answer only on the search results provided. Do not add information from outside the results.
- Structure your answer with a short introduction, key findings, and a brief conclusion.
- When referencing a specific result, cite it by its result number, for example: [1], [2].
- If the search results do not contain enough information to answer the question, say so clearly.
- Write in plain, precise language. No filler phrases."""

def build_planner_user_prompt(user_question: str) -> str:
    return f"Break this research question into 2 or 3 focused sub-questions:\n\n{user_question}"

def build_synthesizer_user_prompt(user_question: str, all_search_results_text: str) -> str:
    return (
        f"Research question: {user_question}\n\n"
        f"Search results:\n{all_search_results_text}\n\n"
        f"Write a structured answer to the research question based on the search results above."
    )

def parse_sub_questions(planner_response: str) -> list[str]:
    """Parses the planner's numbered list into a Python list of sub-questions."""
    sub_questions = []
    for line in planner_response.strip().splitlines():
        cleaned_line = line.strip()
        if not cleaned_line:
            continue
        if cleaned_line[0].isdigit():
            stripped = cleaned_line.lstrip("0123456789").lstrip(".").lstrip(")").strip()
            if stripped:
                sub_questions.append(stripped)
    return sub_questions


# ── agent.py ─────────────────────────────────────────────────────────────────

def _print_step_header(title: str) -> None:
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}\n")

def run_research_agent(model, processor, user_question: str) -> str:
    """Runs the full plan → search → synthesize pipeline."""

    _print_step_header("RESEARCH AGENT STARTING")
    print(f"Question: {user_question}\n")

    # Step 1 — Planner
    _print_step_header("STEP 1: PLANNING")
    print("Asking Gemma 4 to break the question into sub-questions...\n")

    planner_response = generate_response(
        model=model,
        processor=processor,
        user_message=build_planner_user_prompt(user_question),
        system_message=PLANNER_SYSTEM_PROMPT
    )

    print(f"Planner raw output:\n{planner_response}\n")

    sub_questions = parse_sub_questions(planner_response)

    if not sub_questions:
        print("[agent] Planner returned no parseable sub-questions. Aborting.")
        return "Agent failed: planner did not produce valid sub-questions."

    print(f"Parsed {len(sub_questions)} sub-questions:")
    for index, sub_question in enumerate(sub_questions, start=1):
        print(f"  {index}. {sub_question}")

    # Step 2 — Search
    _print_step_header("STEP 2: SEARCHING")

    all_formatted_results = []

    for index, sub_question in enumerate(sub_questions, start=1):
        print(f"[{index}/{len(sub_questions)}] Searching: {sub_question}")

        search_results = search_web(sub_question)

        if not search_results:
            print(f"  No results found for sub-question {index}. Skipping.\n")
            continue

        formatted_results = format_search_results_for_prompt(
            query=sub_question,
            results=search_results
        )
        all_formatted_results.append(formatted_results)
        print(f"  Found {len(search_results)} results.\n")

    if not all_formatted_results:
        print("[agent] All searches returned empty. Aborting.")
        return "Agent failed: no search results were returned for any sub-question."

    combined_search_results = "\n\n".join(all_formatted_results)

    # Step 3 — Synthesizer
    _print_step_header("STEP 3: SYNTHESIZING")
    print("Asking Gemma 4 to synthesize a final answer from search results...\n")

    final_answer = generate_response(
        model=model,
        processor=processor,
        user_message=build_synthesizer_user_prompt(
            user_question=user_question,
            all_search_results_text=combined_search_results
        ),
        system_message=SYNTHESIZER_SYSTEM_PROMPT
    )

    _print_step_header("FINAL ANSWER")
    print(final_answer)

    return final_answer


print("All agent modules loaded.")

## Step 5 — Run the Agent

Change `user_question` to any research topic and run the cell.
The agent will plan, search, and synthesize automatically.

In [ ]:
user_question = "How do transformer models handle long-range dependencies?"

answer = run_research_agent(
    model=model,
    processor=processor,
    user_question=user_question
)

## What's Next

This is Phase 1 — a raw Python agentic pipeline. Natural next steps:

**Phase 2 — Refactor to LangChain**
Replace each component with its LangChain equivalent to understand what the framework actually abstracts:
- `generate_response()` → `HuggingFacePipeline`
- `search_web()` → `DuckDuckGoSearchRun`
- `run_research_agent()` loop → `AgentExecutor` + ReAct
- `prompts.py` templates → `ChatPromptTemplate`

**Enhancement ideas**
- Add memory so the agent can handle follow-up questions
- Enable Gemma 4 thinking mode for harder reasoning questions
- Add a markdown report writer that saves the answer to a `.md` file
- Try the 26B MoE variant (`google/gemma-4-26B-A4B-it`) for deeper answers